# Full Customers Dimension Implementation
This notebook implements the full ingestion and transformation pipeline across **Bronze**, **Silver**, and **Gold** layers for customer dimension data using PySpark and Delta Lake on Databricks.

## 1. Environment Setup & Configuration

In [0]:
%run ../1_setup/utilities

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

# Widgets / Dynamic Parameters
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data source", "products")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data source")

# -------------------------------------------------------------------------
# CENTRALIZED CONFIGURATION (Centralized parameters for easy maintenance)
# -------------------------------------------------------------------------
STORAGE_ACCOUNT = "fmcgaccount.dfs.core.windows.net"
CONTAINER_NAME = "sports-bar-dp"
PARENT_PRODUCTS_DIM_TABLE = "fmcg.gold.dim_products"

# Static Metadata Default Values
DEFAULT_MARKET = "India"
DEFAULT_PLATFORM = "Sports bar"
DEFAULT_CHANNEL = "Aquisition"

# Derived Data Paths and Table Names
base_path = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}/{data_source}/*.csv"
bronze_table_name = f"{catalog}.bronze.{data_source}"
silver_table_name = f"{catalog}.silver.{data_source}"
gold_table_name = f"{catalog}.gold.sb_dim_{data_source}"

print(f"Catalog: {catalog}")
print(f"Data Source: {data_source}")
print(f"Base Path: {base_path}")

## 2. Bronze Layer: Raw Ingestion

In [0]:
# Load raw CSV data with lineage metadata
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

df.display()
df.printSchema()

# Write raw ingestion to Bronze Delta table
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(bronze_table_name)
)

## 3. Silver Layer: Cleaning & Normalization

In [0]:
from itertools import chain
import pyspark.sql.functions as F

# Regex to extract text inside trailing parentheses: e.g., "Item (350g)" -> "350g"
PATTERN = r"\(([^()]+)\)[^()]*$"

# Category to division mapping dictionary
category_to_division_map = {
    "Energy Bars": "Nutrition Bars",
    "Protein Bars": "Nutrition Bars",
    "Granola & Cereals": "Breakfast Foods",
    "Recovery Dairy": "Dairy & Recovery",
    "Healthy Snacks": "Healthy Snacks",
    "Electrolyte Mix": "Hydration & Electrolytes",
}

# Create PySpark Map expression for efficient lookup
mapping_expr = F.create_map([F.lit(x) for x in chain(*category_to_division_map.items())])

# Load bronze table
bronze_df = spark.read.table(bronze_table_name)

silver_df = (
    bronze_df
    # 1. Deduplicate records
    .dropDuplicates(subset=["product_id"])
    
    # 2. Fix specific product ID override
    .withColumn(
        "product_id",
        F.when(
            F.col("product_name") == "SportsBar Oats Cookie Bites ChocoChip (350g)",
            F.lit("25891502")
        ).otherwise(F.col("product_id"))
    )
    
    # 3. Clean up and standardize product name and category spelling/casing
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein"))
    .withColumn(
        "category", 
        F.initcap(F.regexp_replace(F.col("category"), "(?i)Protien", "Protein"))
    )
    
    # 4. Extract variant from product name
    .withColumn("variant", F.regexp_extract(F.col("product_name"), PATTERN, 1))
    
    # 5. Map category to division with "Other" as default fallback
    .withColumn(
        "division",
        F.coalesce(mapping_expr[F.col("category")], F.lit("Other"))
    )
    
    # 6. Generate SHA-256 hash key for product code
    .withColumn("product_code", F.sha2(F.col("product_name").cast("string"), 256))
    
    # 7. Validate product_id format (keep numeric strings, default invalid to '9999999')
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit("9999999"))
    )
    
    # 8. Rename columns to match final target schema
    .withColumnRenamed("product_name", "product")
)

In [0]:

# Write cleaned data to Silver Delta table
(
    silver_df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(silver_table_name)
)

## 4. Gold Layer: Local Dimension Creation

In [0]:
gold_columns = ["product_code","product_id","division","product", "category", "variant"  ]
gold_df = silver_df.select(*gold_columns)

# Write to domain-specific Gold table
(
    gold_df.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(gold_table_name)
)

## 5. Enterprise Integration: Upsert into Parent Dimension

In [0]:
product_column_mapping = {
    "product_code": "product_code",
    "division": "division",
    "category": "category",
    "product": "product",
    "variant": "variant"
}

merge_child_to_parent_dim(
    spark=spark,
    child_table=gold_df,
    parent_table_name=PARENT_PRODUCTS_DIM_TABLE,
    column_mapping=product_column_mapping,
    merge_key="product_code",
    # Specific updates (excludes updating primary key)
    update_set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    },
    # Specific inserts
    insert_values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
)